# 01 – Scene availability
Which Landsat 8, Sentinel-2 and Sentinel-1 images cover the Red River valley during the 2019 flood
(mid-April to early May) and in snow-free reference periods? Used to pick the flood/reference dates
and the AOI, and to document data availability (clouds, coverage, temporal gaps) for the report.

In [1]:
import ee
import pandas as pd

import common as c
import config as cfg

c.init()
cfg.TABLES.mkdir(parents=True, exist_ok=True)

aois = {name: c.get_aoi(name) for name in cfg.AOI_BOXES}

## AOI candidates

In [2]:
m = c.Map()
for aoi_name, aoi in aois.items():
    # addLayer wants a collection, so wrap the AOI box in a FeatureCollection.
    outline = ee.FeatureCollection(aoi)
    m.addLayer(outline, {"color": "red"}, aoi_name)
m

Map(center=[47.9, -97.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

## How to read the tables
`c.scenes_per_day(cfg.Sensor.L8, aoi, window)` gives **one row per day** on which the satellite imaged the AOI:

| column | meaning |
|---|---|
| `date` | acquisition day (UTC) |
| `orbit` | orbit / path number. Sentinel-1 before/after images must have the **same** orbit |
| `n_images` | how many images (Landsat rows, Sentinel-2 tiles, Sentinel-1 frames) that day touch the AOI |
| `aoi_cover_%` | how much of the AOI lies inside these images. Below 100 % = part of the AOI is missing |
| `cloud_%` | mean cloud cover of the **whole scenes** (not only the AOI). Sentinel-1 is radar, so no clouds |

A day is useful when `aoi_cover_%` is close to 100 and `cloud_%` is low.

## During the flood

In [3]:
flood_tables: dict[tuple[str, cfg.Sensor], pd.DataFrame] = {}
for aoi_name, aoi in aois.items():
    for sensor in cfg.Sensor:
        table = c.scenes_per_day(sensor, aoi, cfg.FLOOD_WINDOW)
        flood_tables[(aoi_name, sensor)] = table
        table.to_csv(cfg.TABLES / f"scenes_flood_{aoi_name}_{sensor.name}.csv", index=False)
        print(f"{aoi_name:20s} {sensor.name}: {len(table)} days")

fargo_grandforks     L8: 4 days


fargo_grandforks     S2: 10 days


fargo_grandforks     S1: 4 days


grandforks_pembina   L8: 3 days


grandforks_pembina   S2: 10 days


grandforks_pembina   S1: 4 days


Landsat 8: on 22 April the sky is clear, but the scene covers almost nothing of the Fargo AOI and only part of the Pembina AOI.

In [4]:
flood_tables[("fargo_grandforks", cfg.Sensor.L8)]

,date,orbit,n_images,aoi_cover_%,cloud_%
0,2019-04-15,30,3,100.0,95.7
1,2019-04-22,31,2,0.8,1.1
2,2019-04-24,29,2,46.2,31.2
3,2019-05-08,31,2,2.0,95.5


In [5]:
flood_tables[("grandforks_pembina", cfg.Sensor.L8)]

,date,orbit,n_images,aoi_cover_%,cloud_%
0,2019-04-15,30,2,100.0,96.7
1,2019-04-22,31,2,74.9,1.1
2,2019-05-08,31,2,80.1,95.5


Sentinel-2:

In [6]:
flood_tables[("grandforks_pembina", cfg.Sensor.S2)]

,date,orbit,n_images,aoi_cover_%,cloud_%
0,2019-04-13,12,2,88.7,9.5
1,2019-04-16,55,2,100.0,3.3
2,2019-04-18,12,2,88.2,36.8
3,2019-04-21,55,2,100.0,75.8
4,2019-04-26,55,2,100.0,30.4
5,2019-04-28,12,2,88.1,100.0
6,2019-05-01,55,2,100.0,100.0
7,2019-05-03,12,2,88.0,95.7
8,2019-05-06,55,2,100.0,90.7
9,2019-05-08,12,2,88.4,100.0


Sentinel-1 (radar, no clouds). Two orbits alternate: 63 and 136.

In [7]:
flood_tables[("grandforks_pembina", cfg.Sensor.S1)]

,date,orbit,n_images,aoi_cover_%
0,2019-04-17,63,2,100
1,2019-04-22,136,2,100
2,2019-04-29,63,2,100
3,2019-05-04,136,2,100


## Snow-free reference periods
Here we only count the **good days**: AOI covered by at least 90 % and (for optical) less than 20 % cloud.

In [8]:
MIN_COVER = 90  # %
MAX_CLOUD = 20  # %

REF_WINDOWS = {"autumn2018": cfg.REF_WINDOW_AUTUMN, "summer2019": cfg.REF_WINDOW_SUMMER}

ref_tables: dict[tuple[str, str, cfg.Sensor], pd.DataFrame] = {}
for aoi_name, aoi in aois.items():
    for period, window in REF_WINDOWS.items():
        for sensor in cfg.Sensor:
            table = c.scenes_per_day(sensor, aoi, window)
            ref_tables[(aoi_name, period, sensor)] = table
            table.to_csv(cfg.TABLES / f"scenes_ref_{period}_{aoi_name}_{sensor.name}.csv", index=False)

            good = table[table["aoi_cover_%"] >= MIN_COVER]
            if "cloud_%" in good.columns:
                good = good[good["cloud_%"] < MAX_CLOUD]
            print(f"{aoi_name:20s} {period:11s} {sensor.name}: {len(table):3d} days, good: {len(good)}")

fargo_grandforks     autumn2018  L8:  10 days, good: 1


fargo_grandforks     autumn2018  S2:  23 days, good: 7


fargo_grandforks     autumn2018  S1:  10 days, good: 10


fargo_grandforks     summer2019  L8:  17 days, good: 4


fargo_grandforks     summer2019  S2:  36 days, good: 12


fargo_grandforks     summer2019  S1:  15 days, good: 15


... (2 outputs left out)

grandforks_pembina   autumn2018  S1:  10 days, good: 10


grandforks_pembina   summer2019  L8:  11 days, good: 5


grandforks_pembina   summer2019  S2:  36 days, good: 7


grandforks_pembina   summer2019  S1:  15 days, good: 15


Clear Landsat 8 days in autumn 2018 for the Pembina AOI (path 31 is the same path as the 22 April flood image):

In [9]:
table = ref_tables[("grandforks_pembina", "autumn2018", cfg.Sensor.L8)]
table[table["cloud_%"] < MAX_CLOUD]

,date,orbit,n_images,aoi_cover_%,cloud_%
0,2018-09-03,30,2,100.0,5.1
1,2018-09-10,31,2,81.3,0.0
4,2018-10-12,31,2,79.0,6.3
5,2018-10-21,30,2,100.0,2.3


## Notes / decision

**AOI:** `grandforks_pembina`, clipped to the Landsat 8 footprint of 22 Apr (about Oslo, 48.2° N, to the Canadian border).
Landsat 8 on 22 Apr covers only 0.8 % of `fargo_grandforks`, so that AOI is dropped. By 22 Apr the crest had moved north into this reach.

**Chosen dates**

| | Optical (Landsat 8, path 31) | SAR (Sentinel-1, orbit 136) |
|---|---|---|
| Before | 2018-09-10 (0 % cloud) | median 9 May – 2 Jun 2018 |
| During | 2019-04-22 (1 % cloud) | 2019-04-22 |
| After  | 2019-08-28 (1 % cloud) | 2019-08-20 |

- Landsat 8 and Sentinel-1 on 22 Apr are only about 17 h apart, which is good for the validation.
- All Sentinel-1 images are from orbit 136, so they all see the ground from the same angle.
- Sentinel-1 before: a spring 2018 median, not September, because crop growth causes false change (see notebook 03).
- Before: September 2018. 12 Oct 2018 was the first choice, but its QA band flags 92 % of the area as snow (early snowfall), see notebook 02.

**Data availability / gaps**
- Landsat 8: 22 Apr is the only clear flood date. 15 Apr and 8 May are > 90 % cloud.
- Sentinel-2: clear on 16 Apr (likely snow after the 10–12 Apr blizzard). From 28 Apr to 8 May it is almost fully cloudy.
- Sentinel-1: an image every 5–7 days (17, 22, 29 Apr, 4 May).
- 22 Apr is two weeks after the Fargo peak (8 Apr), but in the study area the river was still close to its peak (Oslo gauge 37.5 ft, major flood stage 36 ft).

**To check:** cloud and snow inside the AOI (cloud % above is for whole scenes) on the chosen dates.

## Check the decision
The chosen dates live in `config.py`. This cell checks that they are really in the tables above and meet the criteria,
so the decision and the evidence can't get out of sync.

In [10]:
MIN_COVER_CHOSEN = 70  # % (the Landsat path-31 scenes cover ~75-80 % of the box)


def row_for(table: pd.DataFrame, day: str) -> dict:
    """The table row of `day` as a dict."""
    rows = table[table["date"] == day]
    assert len(rows) == 1, f"{day} is not in the table"
    return rows.iloc[0].to_dict()


study = cfg.STUDY_AOI
landsat_choices = [
    (cfg.L8_BEFORE, ref_tables[(study, "autumn2018", cfg.Sensor.L8)]),
    (cfg.L8_DURING, flood_tables[(study, cfg.Sensor.L8)]),
    (cfg.L8_AFTER, ref_tables[(study, "summer2019", cfg.Sensor.L8)]),
]
for day, table in landsat_choices:
    row = row_for(table, day)
    assert row["orbit"] == cfg.L8_PATH, f"{day}: wrong path"
    assert row["aoi_cover_%"] >= MIN_COVER_CHOSEN, f"{day}: covers too little"
    assert row["cloud_%"] < MAX_CLOUD, f"{day}: too cloudy"

sentinel1_choices = [
    (cfg.S1_DURING, flood_tables[(study, cfg.Sensor.S1)]),
    (cfg.S1_AFTER, ref_tables[(study, "summer2019", cfg.Sensor.S1)]),
]
for day, table in sentinel1_choices:
    assert row_for(table, day)["orbit"] == cfg.S1_ORBIT, f"{day}: wrong orbit"

print("All chosen dates are in the tables and meet the criteria.")

All chosen dates are in the tables and meet the criteria.
